In [2]:
import polars as pl

df = (
    pl.scan_parquet("./data_arthur/AAPL.OQ/*.parquet")
    .collect()
)

print(df)

shape: (10_876_798, 5)
┌──────────────┬─────────────┬──────────────┬──────────────────┬───────────────┐
│ V1           ┆ trade-price ┆ trade-volume ┆ trade-stringflag ┆ trade-rawflag │
│ ---          ┆ ---         ┆ ---          ┆ ---              ┆ ---           │
│ f64          ┆ f64         ┆ f64          ┆ f64              ┆ f64           │
╞══════════════╪═════════════╪══════════════╪══════════════════╪═══════════════╡
│ 39815.526917 ┆ 85.95       ┆ 175.0        ┆ null             ┆ null          │
│ 39815.539092 ┆ 86.15       ┆ 200.0        ┆ null             ┆ null          │
│ 39815.541686 ┆ 86.15       ┆ 100.0        ┆ null             ┆ null          │
│ 39815.542877 ┆ 86.0        ┆ 174.0        ┆ null             ┆ null          │
│ 39815.543002 ┆ 86.0        ┆ 100.0        ┆ null             ┆ null          │
│ …            ┆ …           ┆ …            ┆ …                ┆ …             │
│ 40178.992194 ┆ 210.7       ┆ 160.0        ┆ null             ┆ null          │
│ 401

In [3]:
df = (
    df
    .drop(["trade-stringflag", "trade-rawflag"])          # enlève les 2 dernières colonnes
    .drop_nulls(subset=["V1", "trade-price", "trade-volume"])  # enlève les lignes avec NaN/null
    .rename({
        "V1": "excel_time",
        "trade-price": "price",
        "trade-volume": "volume",
    })
)

In [5]:
import datetime as dt

origin = dt.datetime(1899, 12, 30)

df = df.with_columns(
    pl.col("excel_time")
    .map_elements(lambda x: origin + dt.timedelta(days=float(x)))
    .alias("datetime")
)

# si tu veux trier :
df = df.sort("datetime")

print(df.head())

shape: (5, 4)
┌──────────────┬───────┬────────┬─────────────────────────┐
│ excel_time   ┆ price ┆ volume ┆ datetime                │
│ ---          ┆ ---   ┆ ---    ┆ ---                     │
│ f64          ┆ f64   ┆ f64    ┆ datetime[μs]            │
╞══════════════╪═══════╪════════╪═════════════════════════╡
│ 39815.526917 ┆ 85.95 ┆ 175.0  ┆ 2009-01-02 12:38:45.595 │
│ 39815.539092 ┆ 86.15 ┆ 200.0  ┆ 2009-01-02 12:56:17.508 │
│ 39815.541686 ┆ 86.15 ┆ 100.0  ┆ 2009-01-02 13:00:01.699 │
│ 39815.542877 ┆ 86.0  ┆ 174.0  ┆ 2009-01-02 13:01:44.610 │
│ 39815.543002 ┆ 86.0  ┆ 100.0  ┆ 2009-01-02 13:01:55.361 │
└──────────────┴───────┴────────┴─────────────────────────┘
